### Imports and model configuration
Import PyTorch and Hugging Face `transformers`, and set the model name to load.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "sshleifer/tiny-gpt2"

/mnt/d/projects/kv-cache-experiment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load the tokenizer and model
Load the tokenizer and causal LM for `MODEL_NAME`, and switch the model to eval mode.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

Loading weights: 100%|██████████| 29/29 [00:00<00:00, 20647.57it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 2)
    (wpe): Embedding(1024, 2)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6, nx=2)
          (c_proj): Conv1D(nf=2, nx=2)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8, nx=2)
          (c_proj): Conv1D(nf=2, nx=8)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=2, out_features=50257, bias=False)
)

In [31]:
import inspect
block0 = model.transformer.h[0]
print(inspect.getsource(type(block0).forward))

    def forward(
        self,
        hidden_states: tuple[torch.FloatTensor] | None,
        past_key_values: Cache | None = None,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.Tensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        use_cache: bool | None = False,
        **kwargs,
    ) -> torch.Tensor:
        residual = hidden_states
        hidden_states = self.ln_1(hidden_states)
        attn_output, _ = self.attn(
            hidden_states,
            past_key_values=past_key_values,
            attention_mask=attention_mask,
            use_cache=use_cache,
            **kwargs,
        )
        # residual connection
        hidden_states = attn_output + residual

        if encoder_hidden_states is not None:
            # add one self-attention block for cross-attention
            if not hasattr(self, "crossattention"):
                raise ValueError(
                    f"If `encoder_

### Verify eval mode
Confirm the model and its dropout layers are all in eval mode (dropout disabled).

In [5]:
block0 = model.transformer.h[0]
print("model.training:", model.training)
print("attention dropout training:", block0.attn.attn_dropout.training)
print("MLP dropout training:", block0.mlp.dropout.training)
print("embedding dropout training:", model.transformer.drop.training)

model.training: False
attention dropout training: False
MLP dropout training: False
embedding dropout training: False


### Tokenize the input prompt
Tokenize the prompt string into input IDs the model can consume.

In [4]:
prompt = "The cat sat on"

inputs = tokenizer(prompt, return_tensors="pt")
inputs

{'input_ids': tensor([[ 464, 3797, 3332,  319]]), 'attention_mask': tensor([[1, 1, 1, 1]])}

### Compute token and position embeddings manually
Look up token and position embeddings and add them to form the initial hidden state `x`.

In [6]:
input_ids = inputs["input_ids"]

position_ids = torch.arange(
    input_ids.shape[1]
).unsqueeze(0)

token_emb = model.transformer.wte(input_ids)
position_emb = model.transformer.wpe(position_ids)

print("Input IDs:")
print(input_ids)
print("Shape:", input_ids.shape)

print("\nToken embeddings:")
print(token_emb)
print("Shape:", token_emb.shape)

print("\nPosition IDs:")
print(position_ids)
print("Shape:", position_ids.shape)

print("\nPosition embeddings:")
print(position_emb)
print("Shape:", position_emb.shape)

x = token_emb + position_emb

print("\nAfter WTE + WPE:")
print(x)
print("Shape:", x.shape)

Input IDs:
tensor([[ 464, 3797, 3332,  319]])
Shape: torch.Size([1, 4])

Token embeddings:
tensor([[[-0.0297,  0.0015],
         [-0.0086, -0.0183],
         [ 0.0185, -0.0232],
         [-0.0192, -0.0276]]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([1, 4, 2])

Position IDs:
tensor([[0, 1, 2, 3]])
Shape: torch.Size([1, 4])

Position embeddings:
tensor([[[ 0.0196, -0.0018],
         [ 0.0356, -0.0140],
         [-0.0013,  0.0035],
         [-0.0238,  0.0099]]], grad_fn=<EmbeddingBackward0>)
Shape: torch.Size([1, 4, 2])

After WTE + WPE:
tensor([[[-0.0101, -0.0003],
         [ 0.0271, -0.0323],
         [ 0.0172, -0.0197],
         [-0.0430, -0.0177]]], grad_fn=<AddBackward0>)
Shape: torch.Size([1, 4, 2])


### Inspect Block 0's first LayerNorm
Look at the `ln_1` module of the first transformer block.

In [7]:
block0.ln_1

LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)

### Apply LayerNorm 1
Run the embeddings `x` through `ln_1` to get the normalized input to attention.

In [8]:
with torch.no_grad():
    x_ln1 = block0.ln_1(x)

### Compare before/after LayerNorm 1
Display `x` and `x_ln1` side by side to see the effect of normalization.

In [9]:
x, x_ln1

(tensor([[[-0.0101, -0.0003],
          [ 0.0271, -0.0323],
          [ 0.0172, -0.0197],
          [-0.0430, -0.0177]]], grad_fn=<AddBackward0>),
 tensor([[[-0.8408,  0.8408],
          [ 0.9944, -0.9944],
          [ 0.9856, -0.9856],
          [-0.9700,  0.9700]]]))

### Run the built-in attention module
Call Block 0's actual `attn` module for reference, to compare against later.

In [10]:
with torch.no_grad():
    attn_output, attn_weights = block0.attn(x_ln1)

### Compute the combined Q/K/V projection
Pass `x_ln1` through `c_attn` to get the combined Q, K, V projection.

In [11]:
with torch.no_grad():
    qkv = block0.attn.c_attn(x_ln1)

print("Input to c_attn:")
print(x_ln1)
print("Shape:", x_ln1.shape)

print("\nOutput of c_attn:")
print(qkv)
print("Shape:", qkv.shape)

print("\nProjection dimensions:")
print("Input features (nx):", block0.attn.c_attn.nx)
print("Output features (nf):", block0.attn.c_attn.nf)
print("split_size:", block0.attn.split_size)

Input to c_attn:
tensor([[[-0.8408,  0.8408],
         [ 0.9944, -0.9944],
         [ 0.9856, -0.9856],
         [-0.9700,  0.9700]]])
Shape: torch.Size([1, 4, 2])

Output of c_attn:
tensor([[[ 0.0159,  0.0317, -0.0063,  0.0054,  0.0524,  0.0035],
         [-0.0188, -0.0374,  0.0074, -0.0064, -0.0620, -0.0042],
         [-0.0186, -0.0371,  0.0074, -0.0064, -0.0614, -0.0041],
         [ 0.0183,  0.0365, -0.0073,  0.0063,  0.0605,  0.0041]]])
Shape: torch.Size([1, 4, 6])

Projection dimensions:
Input features (nx): 2
Output features (nf): 6
split_size: 2


### Inspect the learned c_attn weight and bias
Look at the raw weight matrix and bias of `c_attn`, which packs Q/K/V projections together.

In [12]:
c_attn = block0.attn.c_attn

print("Weight shape:", c_attn.weight.shape)
print("Bias shape:", c_attn.bias.shape)

print("\nFull learned weight matrix W:")
print(c_attn.weight)

print("\nFull learned bias:")
print(c_attn.bias)

Weight shape: torch.Size([2, 6])
Bias shape: torch.Size([6])

Full learned weight matrix W:
Parameter containing:
tensor([[-0.0233, -0.0245, -0.0076, -0.0076, -0.0240, -0.0210],
        [-0.0044,  0.0131, -0.0150, -0.0011,  0.0384, -0.0168]],
       requires_grad=True)

Full learned bias:
Parameter containing:
tensor([0., 0., 0., 0., 0., 0.], requires_grad=True)


### Manually compute the Query projection
Slice out the Q portion of the weight matrix and multiply it by `x_ln1` by hand.

In [16]:
W_Q = block0.attn.c_attn.weight[:, :2]

q = x_ln1 @ W_Q
q

tensor([[[ 0.0159,  0.0317],
         [-0.0188, -0.0374],
         [-0.0186, -0.0371],
         [ 0.0183,  0.0365]]], grad_fn=<UnsafeViewBackward0>)

### Check attention head configuration
Print the number of attention heads and the per-head dimension.

In [14]:
print("Number of attention heads:", block0.attn.num_heads)
print("Head dimension:", block0.attn.head_dim)

Number of attention heads: 2
Head dimension: 1


### Split Q into per-head tensors
Reshape and permute `q` into `[batch, heads, seq, head_dim]` and view each head separately.

In [17]:
num_heads = model.config.n_head
head_dim = model.config.n_embd // num_heads

q_heads = q.view(
    q.shape[0],
    q.shape[1],
    num_heads,
    head_dim
).permute(0, 2, 1, 3)

print("Original q:")
print(q)
print("Shape:", q.shape)

print("\nAfter splitting into heads:")
print(q_heads)
print("Shape:", q_heads.shape)

print("\nHead 1:")
print(q_heads[:, 0, :, :])

print("\nHead 2:")
print(q_heads[:, 1, :, :])

Original q:
tensor([[[ 0.0159,  0.0317],
         [-0.0188, -0.0374],
         [-0.0186, -0.0371],
         [ 0.0183,  0.0365]]], grad_fn=<UnsafeViewBackward0>)
Shape: torch.Size([1, 4, 2])

After splitting into heads:
tensor([[[[ 0.0159],
          [-0.0188],
          [-0.0186],
          [ 0.0183]],

         [[ 0.0317],
          [-0.0374],
          [-0.0371],
          [ 0.0365]]]], grad_fn=<PermuteBackward0>)
Shape: torch.Size([1, 2, 4, 1])

Head 1:
tensor([[[ 0.0159],
         [-0.0188],
         [-0.0186],
         [ 0.0183]]], grad_fn=<SelectBackward0>)

Head 2:
tensor([[[ 0.0317],
         [-0.0374],
         [-0.0371],
         [ 0.0365]]], grad_fn=<SelectBackward0>)


### Manually compute Key and Value projections
Slice the K and V portions of the `c_attn` weight/bias and compute `k` and `v` by hand.

In [18]:
# Block 0 attention
c_attn = block0.attn.c_attn

# Extract the learned K/V projections
W_K = c_attn.weight[:, 2:4]
W_V = c_attn.weight[:, 4:6]

b_K = c_attn.bias[2:4]
b_V = c_attn.bias[4:6]

# Calculate K/V activations
k = x_ln1 @ W_K + b_K
v = x_ln1 @ W_V + b_V

print("x_ln1:", x_ln1.shape)

print("\nW_K:", W_K.shape)
print(W_K)

print("\nk:", k.shape)
print(k)

print("\nW_V:", W_V.shape)
print(W_V)

print("\nv:", v.shape)
print(v)

x_ln1: torch.Size([1, 4, 2])

W_K: torch.Size([2, 2])
tensor([[-0.0076, -0.0076],
        [-0.0150, -0.0011]], grad_fn=<SliceBackward0>)

k: torch.Size([1, 4, 2])
tensor([[[-0.0063,  0.0054],
         [ 0.0074, -0.0064],
         [ 0.0074, -0.0064],
         [-0.0073,  0.0063]]], grad_fn=<AddBackward0>)

W_V: torch.Size([2, 2])
tensor([[-0.0240, -0.0210],
        [ 0.0384, -0.0168]], grad_fn=<SliceBackward0>)

v: torch.Size([1, 4, 2])
tensor([[[ 0.0524,  0.0035],
         [-0.0620, -0.0042],
         [-0.0614, -0.0041],
         [ 0.0605,  0.0041]]], grad_fn=<AddBackward0>)


### Reshape Q, K, V into multi-head format
View and transpose `q`, `k`, `v` into `[batch, heads, seq, head_dim]`.

In [19]:
batch_size, seq_len, d_model = q.shape
num_heads = model.config.n_head
head_dim = d_model // num_heads

Q = q.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
K = k.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)
V = v.view(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)

print("\nQ:")
print(Q)

print("\nK:")
print(K)

print("\nV:")
print(V)

Q: torch.Size([1, 2, 4, 1])
K: torch.Size([1, 2, 4, 1])
V: torch.Size([1, 2, 4, 1])

Q:
tensor([[[[ 0.0159],
          [-0.0188],
          [-0.0186],
          [ 0.0183]],

         [[ 0.0317],
          [-0.0374],
          [-0.0371],
          [ 0.0365]]]], grad_fn=<TransposeBackward0>)

K:
tensor([[[[-0.0063],
          [ 0.0074],
          [ 0.0074],
          [-0.0073]],

         [[ 0.0054],
          [-0.0064],
          [-0.0064],
          [ 0.0063]]]], grad_fn=<TransposeBackward0>)

V:
tensor([[[[ 0.0524],
          [-0.0620],
          [-0.0614],
          [ 0.0605]],

         [[ 0.0035],
          [-0.0042],
          [-0.0041],
          [ 0.0041]]]], grad_fn=<TransposeBackward0>)


### Compute raw attention scores
Compute `Q @ Kᵀ` for each head — the unnormalized attention scores.

In [20]:
scores = Q @ K.transpose(-2, -1)
scores.shape , scores

(torch.Size([1, 2, 4, 4]),
 tensor([[[[-0.0001,  0.0001,  0.0001, -0.0001],
           [ 0.0001, -0.0001, -0.0001,  0.0001],
           [ 0.0001, -0.0001, -0.0001,  0.0001],
           [-0.0001,  0.0001,  0.0001, -0.0001]],
 
          [[ 0.0002, -0.0002, -0.0002,  0.0002],
           [-0.0002,  0.0002,  0.0002, -0.0002],
           [-0.0002,  0.0002,  0.0002, -0.0002],
           [ 0.0002, -0.0002, -0.0002,  0.0002]]]],
        grad_fn=<UnsafeViewBackward0>))

### Apply the causal mask
Build a lower-triangular causal mask and apply it so tokens can't attend to future positions.

In [21]:
causal_mask = torch.tril(
    torch.ones(seq_len, seq_len, dtype=torch.bool)
)

print(causal_mask)

scores_masked = scores.masked_fill(
    ~causal_mask,
    float("-inf")
)

print("\nMasked scores:")
print(scores_masked)

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])

Masked scores:
tensor([[[[-0.0001,    -inf,    -inf,    -inf],
          [ 0.0001, -0.0001,    -inf,    -inf],
          [ 0.0001, -0.0001, -0.0001,    -inf],
          [-0.0001,  0.0001,  0.0001, -0.0001]],

         [[ 0.0002,    -inf,    -inf,    -inf],
          [-0.0002,  0.0002,    -inf,    -inf],
          [-0.0002,  0.0002,  0.0002,    -inf],
          [ 0.0002, -0.0002, -0.0002,  0.0002]]]],
       grad_fn=<MaskedFillBackward0>)


### Softmax into attention weights
Apply softmax to the masked scores to get normalized attention weights.

In [22]:
attention_weights = torch.softmax(
    scores_masked,
    dim=-1
)

print("attention_weights shape:", attention_weights.shape)
print(attention_weights)

attention_weights shape: torch.Size([1, 2, 4, 4])
tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5001, 0.4999, 0.0000, 0.0000],
          [0.3334, 0.3333, 0.3333, 0.0000],
          [0.2500, 0.2500, 0.2500, 0.2500]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4999, 0.5001, 0.0000, 0.0000],
          [0.3332, 0.3334, 0.3334, 0.0000],
          [0.2501, 0.2499, 0.2499, 0.2501]]]], grad_fn=<SoftmaxBackward0>)


### Weighted sum of V
Multiply the attention weights by `V` to get each head's attention output.

In [23]:
attention_output = torch.matmul(
    attention_weights,
    V
)

print("attention_output shape:", attention_output.shape)
print(attention_output)

attention_output shape: torch.Size([1, 2, 4, 1])
tensor([[[[ 0.0524],
          [-0.0048],
          [-0.0237],
          [-0.0026]],

         [[ 0.0035],
          [-0.0003],
          [-0.0016],
          [-0.0002]]]], grad_fn=<UnsafeViewBackward0>)


### Merge the per-head outputs
Transpose and reshape the per-head attention outputs back into a single `[batch, seq, hidden]` tensor.

In [24]:
# [batch, heads, seq, head_dim]
#          ↓
# [batch, seq, heads, head_dim]

attention_combined = attention_output.transpose(1, 2)

print("after transpose:", attention_combined.shape)

# Merge heads and head_dim
attention_combined = attention_combined.reshape(
    batch_size,
    seq_len,
    d_model
)

print("after reshape:", attention_combined.shape)
print(attention_combined)

after transpose: torch.Size([1, 4, 2, 1])
after reshape: torch.Size([1, 4, 2])
tensor([[[ 0.0524,  0.0035],
         [-0.0048, -0.0003],
         [-0.0237, -0.0016],
         [-0.0026, -0.0002]]], grad_fn=<ReshapeAliasBackward0>)


### Apply the output projection
Run the combined attention output through `c_proj`.

In [25]:
attention_projected = block0.attn.c_proj(attention_combined)

print("attention_projected shape:", attention_projected.shape)
print(attention_projected)

attention_projected shape: torch.Size([1, 4, 2])
tensor([[[-2.6997e-04, -1.4627e-04],
         [ 2.4688e-05,  1.3366e-05],
         [ 1.2195e-04,  6.6066e-05],
         [ 1.3534e-05,  7.3433e-06]]], grad_fn=<ViewBackward0>)


### Add the attention residual connection
Add the projected attention output back onto the original embeddings `x`.

In [26]:
x_after_attention = x + attention_projected

print("x shape:", x.shape)
print("attention_projected shape:", attention_projected.shape)

print("\nx_after_attention shape:", x_after_attention.shape)
print(x_after_attention)

x shape: torch.Size([1, 4, 2])
attention_projected shape: torch.Size([1, 4, 2])

x_after_attention shape: torch.Size([1, 4, 2])
tensor([[[-0.0104, -0.0005],
         [ 0.0271, -0.0323],
         [ 0.0173, -0.0197],
         [-0.0430, -0.0177]]], grad_fn=<AddBackward0>)


### Apply LayerNorm 2
Normalize the post-attention hidden state before feeding it into the MLP.

In [27]:
x_ln2 = block0.ln_2(x_after_attention)

print("x_ln2 shape:", x_ln2.shape)
print(x_ln2)

x_ln2 shape: torch.Size([1, 4, 2])
tensor([[[-0.8439,  0.8439],
         [ 0.9944, -0.9944],
         [ 0.9857, -0.9857],
         [-0.9700,  0.9700]]], grad_fn=<NativeLayerNormBackward0>)


### Run the MLP's first linear layer
Apply `c_fc` to project the hidden state up to the MLP's inner dimension.

In [28]:
mlp_fc = block0.mlp.c_fc(x_ln2)

print("mlp_fc shape:", mlp_fc.shape)
print(mlp_fc)

mlp_fc shape: torch.Size([1, 4, 8])
tensor([[[-0.0054,  0.0275,  0.0143,  0.0091, -0.0208, -0.0128, -0.0387,
          -0.0120],
         [ 0.0064, -0.0323, -0.0168, -0.0107,  0.0245,  0.0151,  0.0456,
           0.0141],
         [ 0.0063, -0.0321, -0.0167, -0.0106,  0.0243,  0.0150,  0.0452,
           0.0140],
         [-0.0062,  0.0316,  0.0164,  0.0105, -0.0239, -0.0147, -0.0445,
          -0.0138]]], grad_fn=<ViewBackward0>)


### Apply the GELU activation
Pass the MLP's inner activations through GELU.

In [32]:
mlp_gelu = block0.mlp.act(mlp_fc)

print("mlp_gelu shape:", mlp_gelu.shape)
print(mlp_gelu)

mlp_gelu shape: torch.Size([1, 4, 8])
tensor([[[-0.0027,  0.0140,  0.0072,  0.0046, -0.0102, -0.0063, -0.0188,
          -0.0059],
         [ 0.0032, -0.0158, -0.0083, -0.0053,  0.0125,  0.0076,  0.0236,
           0.0071],
         [ 0.0032, -0.0156, -0.0082, -0.0053,  0.0124,  0.0076,  0.0234,
           0.0071],
         [-0.0031,  0.0162,  0.0083,  0.0053, -0.0117, -0.0073, -0.0215,
          -0.0068]]], grad_fn=<MulBackward0>)


### Project the MLP output back down
Apply `c_proj` to bring the MLP activations back to the model's hidden size.

In [33]:
mlp_projected = block0.mlp.c_proj(mlp_gelu)

print("mlp_projected shape:", mlp_projected.shape)
print(mlp_projected)

mlp_projected shape: torch.Size([1, 4, 2])
tensor([[[-0.0004, -0.0009],
         [ 0.0005,  0.0011],
         [ 0.0005,  0.0011],
         [-0.0004, -0.0010]]], grad_fn=<ViewBackward0>)


### Add the MLP residual connection
Add the MLP output back onto the post-attention hidden state to finish Block 0.

In [34]:
x_after_block0 = x_after_attention + mlp_projected

print("x_after_block0 shape:", x_after_block0.shape)
print(x_after_block0)

x_after_block0 shape: torch.Size([1, 4, 2])
tensor([[[-0.0108, -0.0013],
         [ 0.0275, -0.0312],
         [ 0.0177, -0.0186],
         [-0.0434, -0.0187]]], grad_fn=<AddBackward0>)


### Preview Block 1's first LayerNorm
Quickly check Block 1's `ln_1` output before doing its full forward pass.

In [35]:
block1 = model.transformer.h[1]

x1_ln1 = block1.ln_1(x_after_block0)

print("x1_ln1 shape:", x1_ln1.shape)
print(x1_ln1)

x1_ln1 shape: torch.Size([1, 4, 2])
tensor([[[-0.8308,  0.8308],
         [ 0.9943, -0.9943],
         [ 0.9852, -0.9852],
         [-0.9687,  0.9687]]], grad_fn=<NativeLayerNormBackward0>)


### Manually replicate Block 1's full forward pass
Repeat the whole attention + MLP forward pass for Block 1 by hand, then verify it matches Hugging Face's own `block1` output.

In [36]:
# ============================================================
# BLOCK 1 — manual forward pass
# ============================================================

block1 = model.transformer.h[1]

# ------------------------------------------------------------
# 1. LayerNorm 1
# ------------------------------------------------------------
x1_ln1 = block1.ln_1(x_after_block0)

# ------------------------------------------------------------
# 2. Q, K, V projection
# ------------------------------------------------------------
c_attn1 = block1.attn.c_attn

qkv1 = c_attn1(x1_ln1)

q1, k1, v1 = qkv1.split(d_model, dim=-1)

# ------------------------------------------------------------
# 3. Split into attention heads
# ------------------------------------------------------------
Q1 = q1.view(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

K1 = k1.view(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

V1 = v1.view(
    batch_size, seq_len, num_heads, head_dim
).transpose(1, 2)

# ------------------------------------------------------------
# 4. Attention scores
# ------------------------------------------------------------
scores1 = torch.matmul(
    Q1,
    K1.transpose(-2, -1)
) / (head_dim ** 0.5)

# ------------------------------------------------------------
# 5. Causal mask
# ------------------------------------------------------------
scores1_masked = scores1.masked_fill(
    ~causal_mask,
    float("-inf")
)

# ------------------------------------------------------------
# 6. Softmax
# ------------------------------------------------------------
attention_weights1 = torch.softmax(
    scores1_masked,
    dim=-1
)

# ------------------------------------------------------------
# 7. Weighted sum of V
# ------------------------------------------------------------
attention_output1 = torch.matmul(
    attention_weights1,
    V1
)

# ------------------------------------------------------------
# 8. Combine heads
# ------------------------------------------------------------
attention_combined1 = attention_output1.transpose(1, 2)

attention_combined1 = attention_combined1.reshape(
    batch_size,
    seq_len,
    d_model
)

# ------------------------------------------------------------
# 9. Attention output projection
# ------------------------------------------------------------
attention_projected1 = block1.attn.c_proj(
    attention_combined1
)

# ------------------------------------------------------------
# 10. Attention residual
# ------------------------------------------------------------
x1_after_attention = (
    x_after_block0 +
    attention_projected1
)

# ------------------------------------------------------------
# 11. LayerNorm 2
# ------------------------------------------------------------
x1_ln2 = block1.ln_2(
    x1_after_attention
)

# ------------------------------------------------------------
# 12. MLP: c_fc → GELU → c_proj
# ------------------------------------------------------------
mlp_fc1 = block1.mlp.c_fc(x1_ln2)

mlp_gelu1 = block1.mlp.act(mlp_fc1)

mlp_projected1 = block1.mlp.c_proj(
    mlp_gelu1
)

# ------------------------------------------------------------
# 13. MLP residual
# ------------------------------------------------------------
x_after_block1 = (
    x1_after_attention +
    mlp_projected1
)


# ============================================================
# PRINT SHAPES
# ============================================================

print("x_after_block0      :", x_after_block0.shape)

print("\nx1_ln1              :", x1_ln1.shape)

print("\nq1                  :", q1.shape)
print("k1                  :", k1.shape)
print("v1                  :", v1.shape)

print("\nQ1                  :", Q1.shape)
print("K1                  :", K1.shape)
print("V1                  :", V1.shape)

print("\nscores1             :", scores1.shape)
print("attention_weights1  :", attention_weights1.shape)
print("attention_output1   :", attention_output1.shape)

print("\nattention_combined1 :", attention_combined1.shape)
print("attention_projected1:", attention_projected1.shape)

print("\nx1_after_attention  :", x1_after_attention.shape)

print("\nx1_ln2              :", x1_ln2.shape)
print("mlp_fc1             :", mlp_fc1.shape)
print("mlp_gelu1           :", mlp_gelu1.shape)
print("mlp_projected1      :", mlp_projected1.shape)

print("\nx_after_block1      :", x_after_block1.shape)


# ============================================================
# COMPARE WITH ACTUAL HUGGING FACE BLOCK
# ============================================================

with torch.no_grad():
    actual_block1 = block1(
        x_after_block0
    )[0]

print(
    "\nMaximum difference from Hugging Face Block 1:",
    (x_after_block1 - actual_block1).abs().max().item()
)

print(
    "All close:",
    torch.allclose(
        x_after_block1,
        actual_block1,
        atol=1e-5,
        rtol=1e-5
    )
)

x_after_block0      : torch.Size([1, 4, 2])

x1_ln1              : torch.Size([1, 4, 2])

q1                  : torch.Size([1, 4, 2])
k1                  : torch.Size([1, 4, 2])
v1                  : torch.Size([1, 4, 2])

Q1                  : torch.Size([1, 2, 4, 1])
K1                  : torch.Size([1, 2, 4, 1])
V1                  : torch.Size([1, 2, 4, 1])

scores1             : torch.Size([1, 2, 4, 4])
attention_weights1  : torch.Size([1, 2, 4, 4])
attention_output1   : torch.Size([1, 2, 4, 1])

attention_combined1 : torch.Size([1, 4, 2])
attention_projected1: torch.Size([1, 4, 2])

x1_after_attention  : torch.Size([1, 4, 2])

x1_ln2              : torch.Size([1, 4, 2])
mlp_fc1             : torch.Size([1, 4, 8])
mlp_gelu1           : torch.Size([1, 4, 8])
mlp_projected1      : torch.Size([1, 4, 2])

x_after_block1      : torch.Size([1, 4, 2])

Maximum difference from Hugging Face Block 1: 0.0
All close: True


### Re-confirm eval mode and inspect the full model
Call `model.eval()` again and print the model, ahead of the final LayerNorm and LM head steps.

In [37]:
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 2)
    (wpe): Embedding(1024, 2)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-1): 2 x GPT2Block(
        (ln_1): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=6, nx=2)
          (c_proj): Conv1D(nf=2, nx=2)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=8, nx=2)
          (c_proj): Conv1D(nf=2, nx=8)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((2,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=2, out_features=50257, bias=False)
)

### Apply the final LayerNorm
Run the output of Block 1 through `ln_f`, the model's final layer norm.

In [38]:
x_final = model.transformer.ln_f(x_after_block1)

print("x_final shape:", x_final.shape)
print(x_final)

x_final shape: torch.Size([1, 4, 2])
tensor([[[-0.8530,  0.8530],
         [ 0.9944, -0.9944],
         [ 0.9860, -0.9860],
         [-0.9703,  0.9703]]], grad_fn=<NativeLayerNormBackward0>)


### Project to vocabulary logits
Apply `lm_head` to `x_final` to get logits over the vocabulary for every position.

In [39]:
logits = model.lm_head(x_final)

print("logits shape:", logits.shape)

logits shape: torch.Size([1, 4, 50257])


### Extract the last token's logits
Slice out the logits for the final sequence position, used to predict the next token.

In [40]:
last_logits = logits[:, -1, :]

print("last_logits shape:", last_logits.shape)

last_logits shape: torch.Size([1, 50257])


### Show the top-10 predicted next tokens
List the 10 highest-logit next-token candidates for the last position.

In [41]:
last_logits = logits[:, -1, :]

top_values, top_ids = torch.topk(
    last_logits,
    k=10,
    dim=-1
)

print("Top 10 predictions:\n")

for rank, (token_id, score) in enumerate(
    zip(top_ids[0], top_values[0]),
    start=1
):
    token_id = token_id.item()
    score = score.item()

    token = tokenizer.decode([token_id])

    print(
        f"{rank:2d}. "
        f"id={token_id:5d} "
        f"logit={score: .6f} "
        f"token={token!r}"
    )

Top 10 predictions:

 1. id= 5087 logit= 0.112030 token=' factors'
 2. id=13584 logit= 0.109989 token=' mutual'
 3. id=31685 logit= 0.109046 token=' deflect'
 4. id=21516 logit= 0.107091 token=' Television'
 5. id=40169 logit= 0.104772 token=' boils'
 6. id=44317 logit= 0.100964 token=' Boone'
 7. id=37808 logit= 0.100563 token=' bravery'
 8. id=37416 logit= 0.098656 token=' Wheels'
 9. id=22363 logit= 0.098299 token=' clearer'
10. id=16566 logit= 0.098189 token=' lined'


### Convert logits to probabilities
Apply softmax to get the top-10 next-token probabilities, and confirm they sum to 1.

In [42]:
probs = torch.softmax(last_logits, dim=-1)

top_probs, top_ids = torch.topk(
    probs,
    k=10,
    dim=-1
)

print("Top 10 probabilities:\n")

for rank, (token_id, prob) in enumerate(
    zip(top_ids[0], top_probs[0]),
    start=1
):
    token_id = token_id.item()
    prob = prob.item()

    token = tokenizer.decode([token_id])

    print(
        f"{rank:2d}. "
        f"id={token_id:5d} "
        f"prob={prob:.6f} "
        f"token={token!r}"
    )

print("\nProbability sum:", probs.sum().item())

Top 10 probabilities:

 1. id= 5087 prob=0.000022 token=' factors'
 2. id=13584 prob=0.000022 token=' mutual'
 3. id=31685 prob=0.000022 token=' deflect'
 4. id=21516 prob=0.000022 token=' Television'
 5. id=40169 prob=0.000022 token=' boils'
 6. id=44317 prob=0.000022 token=' Boone'
 7. id=37808 prob=0.000022 token=' bravery'
 8. id=37416 prob=0.000022 token=' Wheels'
 9. id=22363 prob=0.000022 token=' clearer'
10. id=16566 prob=0.000022 token=' lined'

Probability sum: 1.0


### End-to-end sanity check
Run the full model forward pass directly (without cache) and confirm it predicts the same next token as the manual step-by-step derivation above.

In [43]:
prompt = "The cat sat on"

inputs = tokenizer(prompt, return_tensors="pt")

print("Prompt:", prompt)
print("Input IDs:", inputs["input_ids"])
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

with torch.no_grad():
    outputs = model(
        **inputs,
        use_cache=False
    )

logits = outputs.logits

print("Logits shape:", logits.shape)

# Only the last position predicts the next token
next_token_logits = logits[:, -1, :]

next_token_id = torch.argmax(
    next_token_logits,
    dim=-1
)

print("Next token ID:", next_token_id.item())
print("Next token:", repr(tokenizer.decode(next_token_id)))

Prompt: The cat sat on
Input IDs: tensor([[ 464, 3797, 3332,  319]])
Tokens: ['The', 'Ġcat', 'Ġsat', 'Ġon']
Logits shape: torch.Size([1, 4, 50257])
Next token ID: 5087
Next token: ' factors'
